In [2]:
import os
import zipfile
import shutil
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm # tqdm 라이브러리 추가

# ================================================================= #
# 1. 경로 설정 (이 부분만 본인의 환경에 맞게 수정하세요)
# ================================================================= #
SRC_DIR = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/연습용"
DST_RGB_TOP = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/2작기"
DST_RGB_FRONT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/2작기"
DST_THERMAL = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/열화상/0. 원본/2작기"
DST_JSON = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/JSON/2작기"

# 작업 시 임시로 사용할 로컬 경로 (메모리 부족 방지 및 속도 향상)
TEMP_BASE_DIR = r"./temp_work"

# 병렬 작업 수 (보통 4~8 권장, 드라이브 속도에 따라 조절)
MAX_WORKERS = 8

# ================================================================= #
# 2. 유틸리티 함수 (날짜 추출 및 분류 규칙)
# ================================================================= #

def get_date_folder(zip_name):
    """파일명에서 2026-03-12을 찾아 260312 형식으로 반환"""
    try:
        # 'ai_images_' 뒤의 10자리를 날짜로 인식
        date_str = zip_name.split('ai_images_')[1][:10]
        dt = datetime.strptime(date_str, '%Y-%m-%d')
        return dt.strftime('%y%m%d')
    except:
        return "ETC_DATE"

def get_target_path(filename):
    """파일명 규칙에 따라 목적지 폴더 반환"""
    fn = filename.lower()
    if fn.endswith('.json'):
        return DST_JSON
    if 'thermal' in fn:
        return DST_THERMAL
    if 'cam0' in fn or 'cam1' in fn:
        return DST_RGB_TOP
    if 'cam2' in fn:
        return DST_RGB_FRONT
    return None

# ================================================================= #
# 3. 개별 파일 처리 함수 (병렬 실행용)
# ================================================================= #

def process_single_file(args):
    """압축 안의 개별 파일을 추출 후 목적지로 이동 (메모리 절약형)"""
    zip_path, member_name, date_folder = args

    filename = os.path.basename(member_name)
    if not filename: return # 디렉토리인 경우 제외

    target_base = get_target_path(filename)
    if target_base:
        # 최종 목적지 경로 생성
        final_dir = os.path.join(target_base, date_folder)
        os.makedirs(final_dir, exist_ok=True)

        # 압축 파일 내에서 해당 파일만 임시로 추출 후 이동
        with zipfile.ZipFile(zip_path, 'r') as z:
            # 추출 시 파일명 꼬임 방지를 위해 임시 폴더에 추출
            temp_file_path = z.extract(member_name, TEMP_BASE_DIR)
            dest_file_path = os.path.join(final_dir, filename)

            # 이동 (기존 파일 있을 시 덮어쓰기)
            shutil.move(temp_file_path, dest_file_path)
            return True
    return False

# ================================================================= #
# 4. 메인 실행 프로세스
# ================================================================= #

def main():
    overall_start = time.time()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 작업을 시작합니다.")

    if not os.path.exists(SRC_DIR):
        print("에러: SRC_DIR 경로가 올바르지 않습니다.")
        return

    # 압축 파일 목록 확보
    zip_files = [f for f in os.listdir(SRC_DIR) if f.endswith('.zip')]
    print(f"총 {len(zip_files)}개의 압축 파일(15.2GB 규모)을 순차적으로 처리합니다.")

    # tqdm을 사용하여 전체 압축 파일 처리 진행률 표시
    for i, zip_name in enumerate(tqdm(zip_files, desc="전체 진행률")):
        zip_start = time.time()
        zip_path = os.path.join(SRC_DIR, zip_name)
        date_folder = get_date_folder(zip_name)

        tqdm.write(f"\n({i+1}/{len(zip_files)}) 처리 중: {zip_name} -> [폴더: {date_folder}]")

        # 15GB 대용량이므로 전체 압축 해제 대신 파일 목록을 가져와 병렬 처리
        with zipfile.ZipFile(zip_path, 'r') as z:
            member_names = z.namelist()
            # 파일만 필터링
            tasks = [(zip_path, m, date_folder) for m in member_names if not m.endswith('/')]

            tqdm.write(f" - 추출 및 이동 시작 (병렬 스레드: {MAX_WORKERS}개)...")

            # 병렬 실행 (ThreadPoolExecutor 사용으로 I/O 효율 극대화)
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                results = list(executor.map(process_single_file, tasks))

        # 중간 정리 (임시 폴더 비우기)
        if os.path.exists(TEMP_BASE_DIR):
            shutil.rmtree(TEMP_BASE_DIR)
            os.makedirs(TEMP_BASE_DIR)

        zip_end = time.time()
        tqdm.write(f" - {zip_name} 처리 완료: {sum(results)}개 파일 이동")
        tqdm.write(f" - 소요 시간: {zip_end - zip_start:.2f}초")

    # 최종 정리
    if os.path.exists(TEMP_BASE_DIR):
        shutil.rmtree(TEMP_BASE_DIR)

    overall_end = time.time()
    total_min = (overall_end - overall_start) / 60
    print("\n" + "="*50)
    print(f"전체 작업이 종료되었습니다.")
    print(f"총 소요 시간: {total_min:.2f}분")
    print(f"종료 시각: {datetime.now().strftime('%H:%M:%S')}")
    print("="*50)

In [ ]:
if __name__ == "__main__":
    main()

[06:08:44] 작업을 시작합니다.
총 15개의 압축 파일(15.2GB 규모)을 순차적으로 처리합니다.


전체 진행률:   0%|          | 0/15 [00:00<?, ?it/s]


(1/15) 처리 중: ai_images_2026-04-03_2.zip -> [폴더: 260403]
 - 추출 및 이동 시작 (병렬 스레드: 8개)...
 - ai_images_2026-04-03_2.zip 처리 완료: 2400개 파일 이동
 - 소요 시간: 78.23초

(2/15) 처리 중: ai_images_2026-04-03_1.zip -> [폴더: 260403]
 - 추출 및 이동 시작 (병렬 스레드: 8개)...
 - ai_images_2026-04-03_1.zip 처리 완료: 2268개 파일 이동
 - 소요 시간: 62.24초

(3/15) 처리 중: ai_images_2026-04-04_1.zip -> [폴더: 260404]
 - 추출 및 이동 시작 (병렬 스레드: 8개)...
 - ai_images_2026-04-04_1.zip 처리 완료: 2400개 파일 이동
 - 소요 시간: 68.24초

(4/15) 처리 중: ai_images_2026-04-04_2.zip -> [폴더: 260404]
 - 추출 및 이동 시작 (병렬 스레드: 8개)...
 - ai_images_2026-04-04_2.zip 처리 완료: 2316개 파일 이동
 - 소요 시간: 65.60초

(5/15) 처리 중: ai_images_2026-04-05_1.zip -> [폴더: 260405]
 - 추출 및 이동 시작 (병렬 스레드: 8개)...
 - ai_images_2026-04-05_1.zip 처리 완료: 2880개 파일 이동
 - 소요 시간: 88.67초

(6/15) 처리 중: ai_images_2026-04-05_2.zip -> [폴더: 260405]
 - 추출 및 이동 시작 (병렬 스레드: 8개)...
 - ai_images_2026-04-05_2.zip 처리 완료: 1974개 파일 이동
 - 소요 시간: 51.98초

(7/15) 처리 중: ai_images_2026-04-06_1.zip -> [폴더: 260406]
 - 추출 및 이동 시작 (병렬 스레드: 8개

#핵심 파일만 뽑아내기

In [ ]:
import os
import shutil
import time
from datetime import datetime

# ================================================================= #
# 1. 경로 설정
# ================================================================= #
# 원본 데이터가 들어있는 폴더 (260306, 260307 등의 하위 폴더가 있는 곳)
front_view = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/2작기"

# 핵심 파일만 골라서 저장할 폴더
save_folder = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/2작기/260306-260402"

# ================================================================= #
# 2. 핵심 로직 함수
# ================================================================= #

def get_first_images():
    start_time = time.time()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 분석 및 복사를 시작합니다.")

    total_copied = 0

    # 1. 하위 폴더들을 순회 (예: 260306, 260307...)
    sub_folders = [f for f in os.listdir(front_view) if os.path.isdir(os.path.join(front_view, f))]

    for folder_name in sorted(sub_folders):
        current_dir = os.path.join(front_view, folder_name)
        target_dir = os.path.join(save_folder, folder_name)

        # 파일 목록 가져오기 (.jpg만)
        all_files = [f for f in os.listdir(current_dir) if f.lower().endswith('.jpg')]

        if not all_files:
            continue

        # 2. 그룹화 (키: bedXX_YYYYMMDD)
        # 예: 'bed00_20260313' : ['bed00_20260313_124253_cam2.jpg', ...]
        groups = {}
        for f in all_files:
            try:
                # 파일명 규칙: bed00_20260313_124253_cam2.jpg
                parts = f.split('_')
                group_key = f"{parts[0]}_{parts[1]}" # 'bed00_20260313'

                if group_key not in groups:
                    groups[group_key] = []
                groups[group_key].append(f)
            except IndexError:
                continue # 파일명 규칙이 다른 경우 무시

        # 3. 각 그룹에서 시간이 가장 빠른 파일 선택 및 복사
        os.makedirs(target_dir, exist_ok=True)

        folder_copy_count = 0
        for key in groups:
            # 파일 리스트를 정렬하면 시간(124253) 순으로 정렬됨
            sorted_list = sorted(groups[key])
            first_file = sorted_list[0] # 가장 첫 번째(가장 작은 값) 파일

            src_path = os.path.join(current_dir, first_file)
            dst_path = os.path.join(target_dir, first_file)

            # 파일 복사
            shutil.copy2(src_path, dst_path)
            folder_copy_count += 1
            total_copied += 1

        print(f" - 폴더 [{folder_name}] 처리 완료: {folder_copy_count}개 베드 추출됨")

    end_time = time.time()
    print("\n" + "="*50)
    print(f"전체 작업 완료!")
    print(f"총 복사된 파일 수: {total_copied}개")
    print(f"소요 시간: {end_time - start_time:.2f}초")
    print("="*50)

# ================================================================= #
# 3. 실행
# ================================================================= #
if __name__ == "__main__":
    get_first_images()

[15:37:51] 분석 및 복사를 시작합니다.
 - 폴더 [260308] 처리 완료: 89개 베드 추출됨
 - 폴더 [260312] 처리 완료: 38개 베드 추출됨
 - 폴더 [260313] 처리 완료: 81개 베드 추출됨
 - 폴더 [260314] 처리 완료: 82개 베드 추출됨
 - 폴더 [260315] 처리 완료: 90개 베드 추출됨
 - 폴더 [260316] 처리 완료: 86개 베드 추출됨
 - 폴더 [260317] 처리 완료: 93개 베드 추출됨
 - 폴더 [260318] 처리 완료: 93개 베드 추출됨
 - 폴더 [260319] 처리 완료: 93개 베드 추출됨
 - 폴더 [260320] 처리 완료: 92개 베드 추출됨
 - 폴더 [260322] 처리 완료: 11개 베드 추출됨
 - 폴더 [260323] 처리 완료: 10개 베드 추출됨
 - 폴더 [260324] 처리 완료: 91개 베드 추출됨
 - 폴더 [260325] 처리 완료: 91개 베드 추출됨
 - 폴더 [260326] 처리 완료: 90개 베드 추출됨
 - 폴더 [260327] 처리 완료: 70개 베드 추출됨
 - 폴더 [260328] 처리 완료: 67개 베드 추출됨
 - 폴더 [260329] 처리 완료: 92개 베드 추출됨
 - 폴더 [260330] 처리 완료: 28개 베드 추출됨
 - 폴더 [260402] 처리 완료: 88개 베드 추출됨

전체 작업 완료!
총 복사된 파일 수: 1475개
소요 시간: 113.62초
